# 34 — Per-language & specialization findings

Does **specializing per language** beat one multilingual model, and does a **code-mix-designed**
model help the **romanized** tracks? Consolidates four experiments, all on the frozen dev split,
Negative-F1 (sentiment):

1. **Mono vs multi**, full fine-tuning, three encoders (LaBSE, TwHIN-BERT, xlm-roberta).
2. **LoRA vs full fine-tuning** (LaBSE), per language.
3. **Romanized cells** (singlish, tamilish) across encoders + the classical champion.

**Two caveats that gate every number here:**
- **Per-language dev cells are tiny** (~14–68 Negative tickets each), so the CI is wide (~±0.1).
  Read directions, not hairline gaps.
- **Romanized data is synthetic** — Singlish is rule-generated from Sinhala, Tamilish is
  machine-translated (`model-research.md` §5). These cells are an *optimistic upper bound* and do
  not reflect real human-typed romanized text. A code-mix model like TwHIN-BERT would most plausibly
  show its edge on real noisy text we don't have.

In [1]:
import sys, warnings, json, glob
from pathlib import Path
warnings.filterwarnings("ignore")
REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))
import numpy as np, pandas as pd
from swiftbench import splits
pd.set_option("display.width", 200); pd.set_option("display.float_format", lambda v: f"{v:.4f}")
SHA = splits.sha(); print("split sha:", SHA)
LANGS = ["english", "sinhala", "singlish", "tamil", "tamilish"]

def load():
    rows = []
    for f in glob.glob(str(REPO / "ml/reports/runs/sentiment__*__dev.json")):
        d = json.load(open(f))
        if d.get("split_sha") != SHA: continue
        tr = d.get("train_langs"); tr = tr if isinstance(tr, list) else [tr]
        rows.append({"model": d.get("model"), "author": d.get("author", ""),
                     "eval_lang": d.get("eval_lang"), "n_train_langs": len(tr),
                     "negF1": d.get("negative_f1"), "macroF1": d.get("macro_f1")})
    return pd.DataFrame(rows)
runs = load()
print(len(runs), "dev runs on the current split")

split sha: e7b5934392cd
222 dev runs on the current split


## 1. Mono vs multi, full fine-tuning

In [2]:
# Encoder mono-vs-multi per language (full fine-tuning). Deltas within ~+/-0.1 CI per cell.
enc = runs[runs.author.isin(["perlang-multi", "perlang-mono"])]
def cell(model, regime, lang):
    a = "perlang-multi" if regime == "multi" else "perlang-mono"
    m = enc[(enc.model == model) & (enc.author == a) & (enc.eval_lang == lang)]
    return None if m.empty else m.negF1.iloc[0]
for regime in ["multi", "mono"]:
    print(f"\n=== {regime.upper()}-trained, dev Negative-F1 ===")
    tbl = pd.DataFrame({L: {mdl: cell(mdl, regime, L) for mdl in ["labse","twhin-bert","xlmr-base"]}
                        for L in LANGS})
    display(tbl)


=== MULTI-trained, dev Negative-F1 ===


,english,sinhala,singlish,tamil,tamilish
labse,0.6569,0.6713,0.6475,0.6324,0.5512
twhin-bert,0.6533,0.6133,0.5594,0.6259,0.5342
xlmr-base,0.5844,0.4865,0.4889,0.4845,0.4693



=== MONO-trained, dev Negative-F1 ===


,english,sinhala,singlish,tamil,tamilish
labse,0.6667,0.6963,0.6303,0.6803,0.5526
twhin-bert,0.6438,0.5630,0.5373,0.5811,0.5000
xlmr-base,0.5752,0.5333,0.3626,0.5436,0.3364


**Read:** LaBSE leads every cell. For LaBSE, monolingual gives a small edge on native scripts
(tamil, sinhala) but **loses on singlish** — romanized needs the cross-lingual transfer. xlm-roberta
is weakest throughout. All mono-vs-multi deltas sit within the per-cell CI.

## 2. LoRA vs full fine-tuning (LaBSE)

In [3]:
# Full fine-tuning vs LoRA (LaBSE). LoRA run at full-FT hyperparams (lr 2e-5, 3 ep) -- undertrained.
def cell2(author, model, lang):
    m = runs[(runs.author == author) & (runs.model == model) & (runs.eval_lang == lang)]
    return None if m.empty else m.negF1.iloc[0]
lora = pd.DataFrame({L: {
    "ft-multi":   cell2("perlang-multi", "labse", L),
    "ft-mono":    cell2("perlang-mono", "labse", L),
    "lora-multi": cell2("perlang-lora-multi", "labse-lora", L),
    "lora-mono":  cell2("perlang-lora-mono", "labse-lora", L),
} for L in LANGS})
display(lora)

,english,sinhala,singlish,tamil,tamilish
ft-multi,0.6569,0.6713,0.6475,0.6324,0.5512
ft-mono,0.6667,0.6963,0.6303,0.6803,0.5526
lora-multi,0.3821,0.4767,0.3944,0.4242,0.3621
lora-mono,0.3226,0.3223,0.2629,0.3219,0.2739


**Read:** LoRA loses by 0.2–0.3 everywhere, and per-language LoRA (lora-mono) is worst. Caveat:
LoRA ran at full-FT hyperparameters (lr 2e-5) and is undertrained — a fair LoRA test needs a higher
LR (~1e-4) and more epochs. As a drop-in at these settings, it is not competitive.

## 3. Romanized tracks: does a code-mix model help?

In [4]:
# Romanized focus: singlish + tamilish, best config per model, vs the classical champion.
clf = runs[(runs.model == "tfidf-svm")]
def clf_cell(lang, regime):
    if regime == "multi":
        m = clf[(clf.eval_lang == lang) & (clf.n_train_langs == 5)]
    else:
        m = clf[(clf.eval_lang == lang) & (clf.n_train_langs == 1)]
    return None if m.empty else m.negF1.max()
rows = []
for lang in ["singlish", "tamilish"]:
    for mdl in ["labse", "twhin-bert", "xlmr-base"]:
        rows.append({"lang": lang, "model": mdl,
                     "multi": cell(mdl, "multi", lang), "mono": cell(mdl, "mono", lang)})
    rows.append({"lang": lang, "model": "tfidf-svm (classical)",
                 "multi": clf_cell(lang, "multi"), "mono": clf_cell(lang, "mono")})
rom = pd.DataFrame(rows)
rom["best"] = rom[["multi", "mono"]].max(axis=1)
display(rom.sort_values(["lang", "best"], ascending=[True, False]))

,lang,model,multi,mono,best
0,singlish,labse,0.6475,0.6303,0.6475
3,singlish,tfidf-svm (classical),0.6331,0.5954,0.6331
1,singlish,twhin-bert,0.5594,0.5373,0.5594
2,singlish,xlmr-base,0.4889,0.3626,0.4889
7,tamilish,tfidf-svm (classical),0.5468,0.5735,0.5735
4,tamilish,labse,0.5512,0.5526,0.5526
5,tamilish,twhin-bert,0.5342,0.5000,0.5342
6,tamilish,xlmr-base,0.4693,0.3364,0.4693


## Findings

1. **LaBSE is the best encoder on every language, including both romanized tracks.** No specialized
   model or strategy beat it.
2. **TwHIN-BERT (code-mix-designed) does not win on romanized** here — LaBSE beats it on singlish
   (−0.09) and tamilish. It does clearly beat xlm-roberta. Its edge would most likely appear on real
   human-typed code-mixing, which our synthetic data cannot exhibit.
3. **Romanized wants multilingual, not specialization** — every model prefers multi on singlish;
   isolating the track hurts.
4. **Full fine-tuning ≫ LoRA** at matched settings; per-language LoRA is the weakest of all.
5. The one faint pro-specialization signal is **native-script mono** (LaBSE tamil/sinhala), and even
   that is within noise.

**Recommendation:** ship one multilingually fine-tuned LaBSE. Revisit code-mix models and
transliteration only with real human-typed romanized data.